# Conversion des Données de Production Mensuelle de Cuivre en Format Panel

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = "votre_fichier.xlsx"  # Remplacez par le chemin de votre fichier
df = pd.read_excel(file_path)

# Transformer en format long (panel)
df_long = df.melt(id_vars=["Date", "Variable_Identifiant"],  # Remplacez par les colonnes d'identification
                  var_name="Entreprise", 
                  value_name="Production")

# Afficher un aperçu
print(df_long.head())

# Sauvegarder en CSV ou Excel si besoin
df_long.to_csv("panel_data.csv", index=False)


# Téléchargement et Extraction des Coordonnées Géographiques des Sites Miniers à partir de Fichiers KML

In [ ]:
import requests
from xml.etree import ElementTree as ET
import pandas as pd
import folium
from IPython.display import display

def telecharger_fichiers(kml_urls):
    """
    Télécharge les fichiers KML à partir des URL fournies.
    """
    for url in kml_urls:
        response = requests.get(url)
        if response.status_code == 200:
            filename = url.split("/")[-1]
            with open(filename, "wb") as file:
                file.write(response.content)
            print(f"Téléchargé : {filename}")
        else:
            print(f"Erreur lors du téléchargement de : {url}")

def initialiser_namespace():
    """
    Initialise et retourne le dictionnaire des espaces de noms pour le parsing du KML.
    """
    return {'kml': 'http://www.opengis.net/kml/2.2'}

def extraire_locations(root, ns):
    """
    Extrait les coordonnées (latitude, longitude) de toutes les balises <coordinates>
    d'un document KML.
    """
    coordinates = root.findall(".//kml:coordinates", ns)
    locations = []
    for coord in coordinates:
        if coord.text is None:
            continue
        points = coord.text.strip().split()
        for point in points:
            try:
                lon, lat, *_ = point.split(",")
                locations.append((float(lat), float(lon)))
            except ValueError:
                print(f"Format invalide pour le point : {point}")
    return locations

def extract_coordinates(kml_file):
    """
    Extrait les coordonnées d'un fichier KML.
    """
    tree = ET.parse(kml_file)
    root = tree.getroot()
    ns = initialiser_namespace()
    return extraire_locations(root, ns)

def extraire_localites(root, ns):
    """
    Extrait le nom et les coordonnées (latitude, longitude) de chaque localité (Placemark)
    d'un document KML.
    """
    localites = []
    placemarks = root.findall(".//kml:Placemark", ns)
    for placemark in placemarks:
        # Extraire le nom (balise <name>)
        nom_elem = placemark.find("kml:name", ns)
        nom = nom_elem.text if nom_elem is not None else "Nom inconnu"
        
        # Extraire les coordonnées (souvent dans un <Point>)
        coord_elem = placemark.find(".//kml:coordinates", ns)
        if coord_elem is None or coord_elem.text is None:
            continue

        points = coord_elem.text.strip().split()
        for point in points:
            try:
                lon, lat, *_ = point.split(",")
                localites.append((nom, float(lat), float(lon)))
            except ValueError:
                print(f"Format invalide pour le point : {point}")
    return localites

def extract_localities(kml_file):
    """
    Extrait les localités (nom et coordonnées) d'un fichier KML.
    """
    tree = ET.parse(kml_file)
    root = tree.getroot()
    ns = initialiser_namespace()
    return extraire_localites(root, ns)

def afficher_carte(localites):
    """
    Crée une carte interactive avec folium à partir des localités.
    
    Paramètre:
        localites: liste de tuples (nom, latitude, longitude)
    
    Retourne:
        Objet folium.Map
    """
    # Créer un DataFrame à partir des données
    df = pd.DataFrame(localites, columns=["Nom", "Latitude", "Longitude"])
    
    # Définir le centre de la carte : moyenne des latitudes et longitudes
    if not df.empty:
        center = [df["Latitude"].mean(), df["Longitude"].mean()]
    else:
        center = [0, 0]
    
    # Créer la carte
    carte = folium.Map(location=center, zoom_start=6)
    
    # Ajouter un marqueur pour chaque localité
    for _, row in df.iterrows():
        folium.Marker(
            location=[row["Latitude"], row["Longitude"]],
            popup=row["Nom"],
            tooltip=row["Nom"]
        ).add_to(carte)
    
    return carte

def main():
    # Liste des URL de fichiers KML à télécharger
    kml_urls = [
        "https://www.sonami.cl/mapaminero/cobrev49.kml",
        "https://www.sonami.cl/mapaminero/orov16.kml",
        "https://www.sonami.cl/mapaminero/otros13.kml",
        "https://www.sonami.cl/mapaminero/nometalicav47.kml",
        "https://www.sonami.cl/mapaminero/hidrocarburos4.kml",
        "https://www.sonami.cl/mapaminero/infraestructura11.kml"
    ]

    # Télécharger les fichiers KML
    telecharger_fichiers(kml_urls)

    # Choix du fichier KML à analyser (par exemple "cobrev49.kml")
    file_path = "cobrev49.kml"
    
    # Extraction des coordonnées (optionnel)
    coords = extract_coordinates(file_path)
    print("Coordonnées extraites :", coords)

    # Extraction des localités (nom et coordonnées)
    localites = extract_localities(file_path)
    print("Localités extraites :", localites)
    
    # Sauvegarder les localités dans un fichier Excel
    df = pd.DataFrame(localites, columns=["Nom", "Latitude", "Longitude"])
    excel_file = "localites.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"Les localités ont été sauvegardées dans le fichier {excel_file}")
    
    # Création et affichage de la carte interactive
    carte = afficher_carte(localites)
    display(carte)
    
    # Optionnel : sauvegarder la carte dans un fichier HTML
    carte.save("carte_localites.html")
    print("La carte des localités a été sauvegardée dans 'carte_localites.html'.")

if __name__ == "__main__":
    main()


# Fusion des Coordonnées Géographiques des Sites Miniers avec la Base de Production en Panel pour Associer Chaque Site à sa Production et à ses Coordonnées Géographiques

In [ ]:
!pip install python-docx


In [ ]:
!pip install xlrd
!pip install openpyxl

In [25]:
import pandas as pd

In [ ]:
file_path = r"C:\Users\firid\OneDrive\Bureau\Data_Science\PROJETS\CHILI\Données géocodées\données_geocodés\Copie de 11_02_2025__21_06_59_1.xls"

# Read the Excel file taking the first row as the header (this is the default)
df = pd.read_excel(file_path, header=0)

# Optionally, strip extra spaces from column names
df.columns = df.columns.str.strip()

# Display the columns to verify
print(df.columns.tolist())


In [27]:
# Convert the DataFrame from wide to long format.
# "Period" is used as the identifier, and the remaining columns become "Mine" and "Production".
df_long = df.melt(id_vars=["Period"], var_name="Mine", value_name="Production")

In [28]:
# If the production numbers use commas as thousand separators,
# convert them from strings to numeric values by removing the commas.
df_long["Production"] = df_long["Production"].astype(str).str.replace(",", "").astype(float)

In [29]:
# Define the output file path where the long-format DataFrame will be saved
output_file_path = r"C:\Users\firid\OneDrive\Bureau\Data_Science\PROJETS\CHILI\Données géocodées\données_geocodés\df_long.xlsx"

In [30]:
# Save the long-format DataFrame to a new Excel file without the index
df_long.to_excel(output_file_path, index=False)

In [ ]:
# Display the first few rows of the long-format DataFrame
print(df_long.head())

In [32]:
# Load the datasets
df_long = pd.read_excel("df_long.xlsx")
localites = pd.read_excel("localites.xlsx")

In [33]:
# Replace specific values in the 'Mine' column
df_long["Mine"] = df_long["Mine"].replace({
    "Codelco - Salvador": "Salvador",
    "Codelco - Andina": "Andina",
    "Codelco - Teniente": "Teniente"
})

In [34]:
# Remove unwanted rows
df_long = df_long[~df_long["Mine"].isin(["Codelco TOTAL (1)", "Others", "Totals"])]

In [35]:
# Merge with the locations dataset
merged_df = df_long.merge(localites, left_on="Mine", right_on="Nom", how="left")

In [36]:
# Drop the duplicate 'Nom' column after merging
merged_df.drop(columns=["Nom"], inplace=True)

In [37]:
# Save the cleaned and merged dataset to a new Excel file
merged_df.to_excel("merged_data.xlsx", index=False)

In [39]:
# Save the cleaned and merged dataset to a new Excel file
merged_df.to_csv("merged_data.csv", index=False)

In [ ]:
print("Merged file saved as 'merged_data.xlsx'")